In [1]:
import subprocess
import tempfile
from itertools import chain
from pathlib import Path

from IPython.display import Markdown, display
from tqdm.notebook import tqdm

import ghapicache
import githelpers
import release_notes

ReleaseMetadata = release_notes.ReleaseMetadata

In [2]:
%load_ext autoreload
%autoreload 1
%aimport githelpers
%aimport ghapicache
%aimport release_notes

In [3]:
LOCAL_REPO = Path.home() / "Code/celeritas"
githelpers.REPO = LOCAL_REPO

# TODO: implement stashed code
# git = githelpers.GitRunner(local_repo)
# release_notes.GIT = git

In [7]:
CACHED_API = ghapicache.GhApiCache(repo="celeritas")
GHAPI = CACHED_API.api
# CACHED_API.purge()

Failed to load cache: [Errno 2] No such file or directory: 'data/ghapicache-celeritas-project-celeritas.json'


In [8]:
int(GHAPI.limit_rem)

5000

# Pre-loading data

## All pull requests

In [ ]:
import asyncio

# PR parser uses local data
PR_PARSER = release_notes.PullRequestLogParser(Path("data/github-sha-map.json"))


async def parallel_load_contributions(
    cached: release_notes.GhApiCache, pull_ids: list[int], limit: int = 8
):
    semaphore = asyncio.Semaphore(limit)

    async def load(pr_id):
        async with semaphore:
            return await release_notes.async_load_contributions(cached, pr_id)

    tasks = [asyncio.create_task(load(pr_id)) for pr_id in pull_ids]
    contributions = []
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        contributions.append(await task)
    return contributions


async def load_contributions(release_md: release_notes.ReleaseMetadata):
    prs = release_notes.PullRequestRange(release_md, PR_PARSER)
    contributions = await parallel_load_contributions(CACHED_API, prs.pull_ids)
    return release_notes.merge_contributions(contributions)


In [ ]:
# Load all commits from all time
develop_sha = githelpers.git_rev_parse("upstream/develop")
all_md = ReleaseMetadata(
    release=f"develop-{develop_sha}",
    merge_bases=["v0.0.0"],
    target_branch=develop_sha,
)
all_prs = release_notes.PullRequestRange(all_md, PR_PARSER)
contributions = await parallel_load_contributions(CACHED_API, all_prs.pull_ids)
ALL_CONTRIB = release_notes.merge_contributions(contributions)
CACHED_API.flush()

Ignore 'Format code base (clang-format version 11.0.1)' (7093314f26c023d664e1ff65a154fd5b4c699c03): old clang-format
Ignore 'Fix linking to CUDA toolkit when using VecGeom' (34e8a82fee4f6a7ede5d8c574aaa57c9acdf0a6d): hotfix vecgeom
Ignore 'Add option to use spline-interpolated energy loss (#1496)' (d2871473f447f27b97b43876258f87bb17c08746): weird github pull glitch from 'new feature'


  0%|          | 0/2179 [00:00<?, ?it/s]

Shut down early: contributions are incomplete
Saved cache to: data/ghapicache-celeritas-project-celeritas.json


In [11]:
CACHED_API.flush()

Saved cache to: data/ghapicache-celeritas-project-celeritas.json


## Teams and members

In [ ]:
# Institutional addresses used for printing
INST_ADDRESS = {
    "CERN": "CERN, Geneva, Switzerland",
    "ORNL": "Oak Ridge National Laboratory, Oak Ridge, TN, USA",
    "U Warwick": "Department of Physics, University of Warwick, Coventry, United Kingdom",
    "FNAL": "Fermi National Accelerator Laboratory, Batavia, IL, USA",
    "ANL": "Argonne National Laboratory, Lemont, IL, USA",
    "LBNL": "Lawrence Berkeley National Laboratory, Berkeley, CA, USA",
    "UVA": "University of Virginia, Charlottesville, VA, USA",
    "Wayne State U": "Wayne State University, Detroit, MI, USA",
    "BNL": "Brookhaven National Laboratory, Upton, NY, USA",
}

In [ ]:
ALL_CONTRIB.author.keys()

In [ ]:
# Load teams
TEAMS: dict[str, release_notes.Team] = {}
team_metadata = await CACHED_API.async_teams()
team_results = await asyncio.gather(
    *(release_notes.async_get_team(CACHED_API, t) for t in team_metadata)
)
for t, team in zip(team_metadata, team_results):
    TEAMS[t["name"]] = team

CORE_TEAM_NAMES = {"core", "core-advisor", "code-lead"}
team_members = frozenset(chain(*(TEAMS[t].members for t in CORE_TEAM_NAMES)))
org_members = frozenset(m["login"] for m in await CACHED_API.async_org_members())
TEAMS["affiliates"] = release_notes.Team(
    description="Associated with Celeritas but not core members",
    members=(org_members - team_members),
)

# Initialize the user cache
USER_CACHE = release_notes.UserCache(
    CACHED_API, LOCAL_REPO / "scripts/release/contributors.jsonl"
)
await asyncio.gather(
    *(
        USER_CACHE.aload(login)
        for login in sorted(ALL_CONTRIB.author.keys() | team_members)
    )
)

In [ ]:
!open {USER_CACHE.json_path}

In [ ]:
USER_CACHE.flush()

# Print authors and members

## Active members

This is to be used for crediting in presentations, etc.

In [ ]:
text = []
for team, title in [
    ("code-lead", "Code lead"),
    ("core", "Core members"),
    ("core-advisor", "Core advisors"),
    ("affiliates", "Affiliates"),
]:
    text.append(f"## {title}")
    m = [USER_CACHE[username] for username in TEAMS[team].members]
    for member in sorted(m, key=release_notes.get_last_name):
        if member.name is None:
            print(f"Missing metadata: {member!r}")
            continue
        text.append("- " + release_notes.format_user(member))
    text.append("")

display(Markdown("\n".join(text)))

In [ ]:
collaborators = [USER_CACHE[username] for username in USER_CACHE.users]

In [ ]:
collaborators[0]._asdict()

In [ ]:
import csv

# Write collaborators to CSV
with open("collaborators.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["last", "first", "orcid", "institution"])

    for collab in sorted(collaborators, key=release_notes.get_last_name):
        # Split name into first and last
        if collab.name:
            name_parts = collab.name.strip().split()
            if len(name_parts) > 1:
                last = name_parts[-1]
                first = " ".join(name_parts[:-1])
            else:
                last = name_parts[0]
                first = ""
        else:
            first = ""
            last = ""

        writer.writerow([last, first, collab.orcid or "", collab.institute or ""])

print(f"Wrote {len(collaborators)} collaborators to collaborators.csv")

## Contributors since the last major release for presentations

In [ ]:
latest_major_md = ReleaseMetadata(
    release="recent-develop",
    merge_bases=["v0.7.0-dev"],
    target_branch="upstream/develop",
)

prs = release_notes.PullRequestRange(latest_major_md, PR_PARSER)
contributions = await parallel_load_contributions(CACHED_API, prs.pull_ids)
merged_contrib = release_notes.merge_contributions(contributions)

text = ["# Current contributors"]
current_contrib = set()
for username in merged_contrib.author:
    m = USER_CACHE[username]
    text.append(f"- {m.name} (@{m.github})")
    current_contrib.add(m.github)
text.extend(["", "# Past contributors"])
for username in ALL_CONTRIB.author:
    if username not in current_contrib:
        m = USER_CACHE[username]
        text.append(f"- {m.name} (@{username})")
# display(Markdown("\n".join(text)))
print("\n".join(text))

In [ ]:
display(Markdown("\n".join(text)))

In [ ]:
CACHED_API.flush()

## CHEP authors and contributors

In [ ]:
for username in tqdm(set(ALL_CONTRIB.author) - set(USER_CACHE.users)):
    print(USER_CACHE[username])

In [ ]:
def ordered_uniquify(x: list):
    result: list = []
    seen = set()

    for v in x:
        if v in seen:
            continue
        result.append(v)
        seen.add(v)

    return result


def print_chep_authors(contributors, institutes):
    """Print CHEP author blocks given contributors and ordered institute list."""
    institute_map = {inst: i for i, inst in enumerate(institutes, start=1)}
    author_blocks = []

    for c in contributors:
        parts = c.name.split()
        if len(parts) > 1:
            first = " ".join(parts[:-1])
            last = parts[-1]
        else:
            first = ""
            last = parts[0]

        inst_index = institute_map[c.institute]
        inst_suffix = f"\\inst{{{inst_index}}}"
        orcid_suffix = f"\n  \\orcidlink{{{c.orcid}}}%" if c.orcid else ""

        author_blocks.append(
            f"  \\firstname{{{first}}} \\lastname{{{last}}}{inst_suffix}%{orcid_suffix}"
        )

    if not author_blocks:
        print("  No authors found")
        return

    print("\n  %%\n  \\and\n".join(author_blocks))


def print_chep_institutes(institutes):
    """Print institute addresses joined with \and."""
    addresses = []
    for institute in institutes:
        addresses.append(INST_ADDRESS[institute])

    if not addresses:
        print("  No institutes found")
        return

    print("\n  \\and\n".join(f"  {addr}" for addr in addresses))

In [ ]:
chep: list[release_notes.UserInfo] = []
for team in ["code-lead", "core", "core-advisor"]:
    users = [USER_CACHE[username] for username in TEAMS[team].members]
    users.sort(key=lambda u: u.name.split()[-1])
    for u in users:
        if u.institute is None:
            print(f"missing insitute: {u}")
            continue
        chep.append(u)

chep

print({u.institute for u in chep} - set(INST_ADDRESS))

In [ ]:
institutes = ordered_uniquify(["CERN", "ORNL"] + [u.institute for u in chep])
print_chep_authors(chep, institutes)
print_chep_institutes(institutes)

# Release process

- Merge base should be all commits *already* released (skip documenting)
- Target branch is the one where the release candidate is
- Previous major branch allows all "v.x" contributors to be credited

In [ ]:
# Major release
major_md = release_notes.ReleaseMetadata(
    release="0.7.0", merge_bases=["v0.6.0", "v0.6.3"], target_branch="upstream/develop"
)

# Minor release
minor_md = release_notes.ReleaseMetadata(
    release="0.7.1", merge_bases=["v0.7.0"], target_branch="upstream/develop"
)

# Release candidate (atypical)
rc_md = release_notes.ReleaseMetadata(
    release="0.7.0-rc.1",
    # merge_bases=["v0.6.0", "v0.6.3"],
    merge_bases=["v0.7.0-dev^"],
    target_branch="upstream/develop",
)

## Release note generation

In [ ]:
if 0:
    # Backport release:
    release_md = minor_md
    note_body = """
Version {release} is a minor bug fix update.

- Support CUDA 13
- Allow initialization of Geant4 integration before run manager
"""
elif False:
    # Major release
    release_md = major_md
    note_body = """
Version {release} is a major update to Celeritas featuring many new capabilities.

-

A few minor features are noteworthy:

- 

Notable bug fixes include:

- 

Some interfaces have been removed:

- 

Known deficiencies:

- 
"""

else:
    # 0.7
    release_md = major_md
    note_body = """
Version {release} is a long-overdue major update to Celeritas featuring many new capabilities.

- Optical physics is now fully supported, including surface properties, Geant4
  process integration, and Mie/WLS processes.
- A LArSoft plugin that stands in for PDFastSimPar uses full-fidelity MC transport
  on CPU or GPU for optical photon transport from simulated energy deposition.
- The EM tracking loop is now asynchronous (relying on built-in CUDA asynchronous
  streaming and events rather than an additional control thread).
- ORANGE geometry navigation is now substantially faster and now
  completely supports Geant4 quadric solids.
- G4 tracking manager integration is now set up automatically rather than
  requiring BeginOfRun/EndOfRun actions.

Major known deficiencies include:

- Surface normal calculations for VecGeom is unsupported, so optical physics
  will be wrong if any surface physics is used.
- The behavior of reflectivity/efficiency grids acting as pure absorbers
  around sensitive detector regions (i.e., black boxes for splicing)
  currently differs from Geant4.
- Tessellated solids are not supported when using ORANGE.
"""

In [ ]:
prs = release_notes.PullRequestRange(release_md, PR_PARSER)
sorted_pulls = release_notes.SortedPulls(CACHED_API)

pull_tasks = [
    asyncio.create_task(sorted_pulls.async_add(pr_id)) for pr_id in prs.pull_ids
]
contribution_tasks = [
    asyncio.create_task(release_notes.async_load_contributions(CACHED_API, pr_id))
    for pr_id in prs.pull_ids
]
contributions = []
for pull_task, contribution_task in tqdm(
    zip(pull_tasks, contribution_tasks), total=len(prs.pull_ids)
):
    try:
        await pull_task
        contributions.append(await contribution_task)
    except Exception as e:
        print(f"Error adding PR: {e}")

reviewers = release_notes.merge_contributions(contributions).reviewer
for login in tqdm(reviewers):
    await USER_CACHE.aload(login)

In [ ]:
CACHED_API.flush()

In [ ]:
def make_notes(cls, release_md, note_body):
    prev = release_md.merge_bases[0]
    other = release_md.merge_bases[1:]
    if not other:
        change_str = f"Changes since {prev} follow."
    else:
        # NOTE: this is redundant if prev is a merge base of other
        assert len(other) == 1
        change_str = (
            f"Changes since {prev}, excluding those released in {other[0]}, follow."
        )

    notes = cls(release_md, note_body)
    notes.paragraph(change_str)
    notes.sorted_pulls(sorted_pulls)
    notes.reviewers(reviewers, USER_CACHE)
    notes.changelog_line("celeritas-project", "celeritas")

    return notes

In [ ]:
rst_notes = make_notes(release_notes.RstNotes, release_md, note_body)

with githelpers.open_pbcopy() as pb:
    rst_notes.write(pb)

print("Release notes copied to clipboard")
subprocess.run(["open", str(githelpers.REPO / "doc/release-history")])

## Create github release

Make sure the release history above is *first* merged into upstream.
For backport releases, *also* ensure it is **cherry-picked into the backport branch**.

First, tag the release locally and update the target to the tagged commit.

In [ ]:
release_md

In [ ]:
if True:
    print(githelpers.git("remote", "update"))
    subject = githelpers.git("log", "-1", r"--format=%s", release_md.target_branch)[0]
    if not subject.startswith("Release v" + release_md.release):
        print(
            f"WARNING: expected last commit subject to include release, but subject line is '{subject}'"
        )

    release_md = release_notes.create_local_tag(release_md)
else:
    release_md.target_branch = githelpers.gitz(
        "log", "-1", "-z", "--format=%H", f"v{release_md.release}"
    )[0]
print(release_md)

Then create a gzipped in-memory archive of the repository. Decompress and run `cmake -P cmake/CgvFindVersion.cmake` to check the version is exported correctly.

In [ ]:
tgz_data = githelpers.git_archive_tgz(release_md.target_branch)
tgz_name = f"{CACHED_API.repo}-{release_md.release}.tar.gz"

with tempfile.TemporaryDirectory(delete=False) as tmpdir:
    with open(Path(tmpdir) / tgz_name, "wb") as f:
        f.write(tgz_data)
    subprocess.run(["open", tmpdir])

In [ ]:
subprocess.run(
    ["cmake", "-P", "cmake/CgvFindVersion.cmake"],
    cwd=Path(tmpdir) / f"{CACHED_API.repo}-{release_md.release}",
)

Now generate the PDF documentation and check it.

In [ ]:
build_dir = LOCAL_REPO / "build"


def run_build(*args):
    return subprocess.run(args, cwd=build_dir, check=True)


githelpers.git("checkout", "--detach", release_md.target_branch)
run_build("cmake", "-UCeleritas_GIT_DESCRIBE", ".")
run_build("ninja", "doc_pdf")
pdf_doc = build_dir / "doc" / "latex" / "Celeritas.pdf"
assert pdf_doc.is_file()
!open {pdf_doc}

Create the github release draft and upload the tarball and PDF documentation

In [ ]:
GHAPI = CACHED_API.api

markdown_notes = make_notes(release_notes.MarkdownNotes, release_md, note_body)

assert release_md.release is not None
gh_release = release_notes.find_release(GHAPI, release_md.release)
if not gh_release:
    print("Creating release")
    gh_release = release_notes.create_release(GHAPI, release_md, str(markdown_notes))
    tgz_asset = release_notes.upload_asset(CACHED_API, gh_release, tgz_data, tgz_name)
    doc_asset = release_notes.upload_asset(
        CACHED_API,
        gh_release,
        pdf_doc,
        f"celeritas-user-manual-{release_md.release}.pdf",
    )
else:
    # Restarting from later:
    # WRONG though
    tgz_data = release_notes.get_tarball(CACHED_API, gh_release)

In [ ]:
sha256_hash = release_notes.prepare_spack_update("celeritas", release_md, tgz_data)

In [ ]:
display(
    Markdown(f"""
## Time to publish!

1. Visit [the github release page]({gh_release["html_url"]})
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is {sha256_hash}
4. Double check the name and release notes
5. Edit and publish!
""")
)

# Push to zenodo

In [ ]:
import zenodoapi

%aimport zenodoapi

In [ ]:
# Load the Zenodo token
token_path = Path.home() / ".config/zenodo-token"
with open(token_path) as f:
    zenodo_token = f.read().strip()

zenodo = zenodoapi.Zenodo(zenodo_token)
zenodo.api_url = "https://zenodo.org/api/"

In [ ]:
def download_tarball(gh_release):
    asset = gh_release["assets"][0]
    name = asset["name"]
    content = CACHED_API.download_file(asset["url"], ext=Path(name).suffix)
    assert content is not None
    return (content, name)

In [ ]:
make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=USER_CACHE, teams=TEAMS)

## Minor release or update

This assumes the major release has already been published upstream into Zenodo.

In [ ]:
release_md.target_branch = "v0.7.0"

In [ ]:
# assert not release_md.is_major()
major, minor, patch = release_md.as_version()
# release_md = release_notes.ReleaseMetadata.from_comprehensive_version(minor, patch)
major_dep = zenodo.find_deposition(f"Celeritas {major}.{minor}")

In [ ]:
zenodo.get_deposition(major_dep.get_latest_version().id)

In [ ]:
prs = release_notes.PullRequestRange(release_md, PR_PARSER)
contributions = await parallel_load_contributions(CACHED_API, prs.pull_ids)
merged_contrib = release_notes.merge_contributions(contributions)

In [ ]:
gh_release = release_notes.find_release(GHAPI, release_md.release)
new_md = make_zenodo_md(merged_contrib, release_md, gh_release)

In [ ]:
new_md

In [ ]:
# Get the Zenodo metadata
from requests import HTTPError

old_md = major_dep.data["metadata"]
# Only update editors, not team members
old_contrib = new_md["contributors"]
new_contrib = [u for u in old_contrib if u["type"] == "Editor"]
old_contrib = [u for u in old_contrib if u["type"] != "Editor"]
new_md["contributors"] = new_contrib + old_contrib
# Don't change the title
new_md["title"] = major_dep.data["metadata"]["title"]
# Add the body
if not release_md.is_major():
    new_md["description"] = "\n\n".join([
        old_md["description"],
        f"<h1>Version {release_md.release}</h1>",
        new_md["description"],
    ])
try:
    new_vers = major_dep.create_new_version()
except HTTPError as e:
    # Maybe the version already exists
    if e.response.status_code == 400:
        print("Draft may already exist:", e)
        new_vers = major_dep.get_latest_draft()
    else:
        raise
try:
    new_vers.update(new_md)
except HTTPError as e:
    # Something else went wrong? Try to upload the tarball
    print("Draft may already exist:", e)

# Upload the release
(content, name) = download_tarball(gh_release)
new_vers.upload(content, name)
# The old tarball may still be there (this is buggy) so delete it
new_vers.refresh()
for file in new_vers.get_files():
    if file.filename != name:
        try:
            file.delete()
        except Exception as e:
            print(f"Failed to delete {file.filename}: {e}")
new_vers.refresh()

## Major release

In [ ]:
release_md

In [ ]:
release_md.as_version()

In [ ]:
assert release_md is not None and release_md.is_major()
major, minor, patch = release_md.as_version()
# release_md = release_notes.ReleaseMetadata.from_comprehensive_version(major, minor)
title = f"Celeritas {major}.{minor}"

In [ ]:
gh_release = release_notes.find_release(GHAPI, release_md.release)

In [ ]:
ALL_CONTRIB

In [ ]:
prs = release_notes.PullRequestRange(release_md, PR_PARSER)
contributions = await parallel_load_contributions(CACHED_API, prs.pull_ids)
ALL_CONTRIB = release_notes.merge_contributions(contributions)

In [ ]:
make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=USER_CACHE, teams=TEAMS)
zmd = make_zenodo_md(ALL_CONTRIB, release_md, gh_release)

In [ ]:
zmd

In [ ]:
zmd["title"] = title

In [ ]:
# Create the Zenodo deposition
deposition = zenodo.create_deposition(zmd)

In [ ]:
zmd

In [ ]:
# Upload the release tarball
deposition.upload(artifact_tgz, f"celeritas-{release_md.release}.tar.gz")

print(deposition.html)
print("""
IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses
""")